# 04 · Evaluación e iteración

El objetivo de este notebook es cerrar el taller con criterio: no basta con que el sistema “suene bien”, también tiene que recuperar bien y abstenerse cuando toca.


In [ ]:
import csv
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")
sys.path.insert(0, str((Path("..") / "app").resolve()))

from settings import Settings
from rag.pipeline import RAGPipeline

settings = Settings.from_env()
pipeline = RAGPipeline(settings)
csv_path = Path("../evaluation/questions.csv")

with csv_path.open("r", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

rows

## Paso 1 · Ejecutar el conjunto de preguntas

In [ ]:
results = []
for row in rows:
    answer = pipeline.ask(row["question"], top_k=settings.top_k)
    results.append({
        "id": row["id"],
        "question": row["question"],
        "expected_source": row.get("expected_source", ""),
        "should_abstain": row["should_abstain"],
        "sources": answer["sources"],
        "answer": answer["answer"],
    })

results

## Paso 2 · Localizar fallos

Busca patrones:
- fuente incorrecta,
- respuesta correcta pero mal fundamentada,
- falta de abstención,
- retrieval bueno pero redacción mala,
- retrieval malo desde el principio.

In [ ]:
for item in results:
    print("=" * 100)
    print(item["id"], item["question"])
    print("Fuentes:", item["sources"])
    print(item["answer"])
    print()

## Paso 3 · Decidir la siguiente iteración

Escribe un cambio que probarías primero: `chunk_size`, `chunk_overlap`, `top_k`, prompt o documentos.

In [ ]:
siguiente_iteracion = ""
siguiente_iteracion